# 🚀 ColabDrive Downloader & Merger
Công cụ tải video từ mọi nguồn (YouTube, Facebook, TikTok, Torrent/Magnet, Direct Link, luồng trực tiếp m3u8/mpd, danh sách phân mảnh .m4s/.ts) và ghép video tốc độ cao trực tiếp vào **Google Drive**.

**Đặc điểm nổi bật:**
- Tốc độ tải và xử lý cực nhanh nhờ hạ tầng mạng máy chủ Google Colab.
- Lưu trữ trực tiếp vào Google Drive để không bị mất dữ liệu khi ngắt kết nối.
- Giao diện Form trực quan dễ sử dụng, ẩn toàn bộ code phức tạp.
- Hỗ trợ tải phân mảnh video (m4s, ts...) từ danh sách link và tự động gộp chuẩn xác bằng `ffmpeg`.
- Hỗ trợ quay ngược thời gian tải video xem lại (Catchup/Replay) từ luồng Live IPTV.
- Hỗ trợ gộp ghép nhiều video khác nhau có sẵn trên Drive hoặc mới tải về.

**Hướng dẫn:** Chạy từng ô từ trên xuống dưới bằng nút ▶️ bên trái.


In [ ]:
#@title 📦 Bước 1: Cài đặt công cụ và thư viện nền tảng (Chạy 1 lần)
#@markdown Chạy ô này để tải và cài đặt các thư viện cần thiết (`yt-dlp`, `ffmpeg`, `aria2`).

import os
import sys

print("⏳ Đang cài đặt/cập nhật thư viện yt-dlp...")
os.system("pip install --force-reinstall -q yt-dlp")

print("⏳ Đang cài đặt aria2c và ffmpeg...")
os.system("apt-get update -y -q && apt-get install -y -q ffmpeg aria2")

print("✅ Đã cài đặt hoàn tất các công cụ!")


In [ ]:
#@title 💾 Bước 2: Kết nối Google Drive
#@markdown Chạy ô này để liên kết với tài khoản Google Drive của bạn. File tải về sẽ được lưu trữ trực tiếp trên Drive.

from google.colab import drive
import os

print("⏳ Vui lòng nhấn vào liên kết xuất hiện để cấp quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("✅ Đã kết nối Google Drive thành công!")
print("📁 Thư mục gốc Drive của bạn: /content/drive/MyDrive/")


In [ ]:
#@title ⬇️ Bước 3: Tải Video từ một Link (YouTube, Facebook, Torrent, m3u8, mpd, Direct Link...)
#@markdown Dán link video cần tải vào bên dưới. Công cụ sẽ tự nhận dạng định dạng và tải xuống với tốc độ tối đa.

VIDEO_LINK = "" #@param {type:"string"}
DRIVE_SAVE_PATH = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
CUSTOM_HEADERS = "" #@param {type:"string"}
USE_ARIA2 = True #@param {type:"boolean"}

import os
import re
import glob
import shutil
import time
import urllib.parse
import yt_dlp

def parse_headers(headers_str):
    headers = {}
    if not headers_str.strip():
        return headers
    for line in headers_str.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':', 1)
            headers[k.strip()] = v.strip()
    return headers

def format_size(bytes_val):
    if not bytes_val: return "N/A"
    bytes_val = float(bytes_val)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_val < 1024.0:
            return f"{bytes_val:.1f} {unit}"
        bytes_val /= 1024.0
    return f"{bytes_val:.1f} PB"

if not VIDEO_LINK.strip():
    raise ValueError("❌ Bạn chưa nhập link video!")

# Tạo thư mục tạm và thư mục đích
temp_dir = "/content/temp_download"
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir, exist_ok=True)
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print("🔍 Đang chuẩn bị phân tích link...")

is_torrent = VIDEO_LINK.startswith('magnet:?') or VIDEO_LINK.endswith('.torrent') or '.torrent?' in VIDEO_LINK

if is_torrent:
    # --- Tải torrent dùng aria2 ---
    print("📦 Phát hiện link Torrent / Magnet. Bắt đầu tải...")
    trackers = ""
    try:
        import urllib.request
        req = urllib.request.Request('https://cf.trackerslist.com/best.txt', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as response:
            trackers_list = response.read().decode('utf-8').strip().split('\n')
            trackers = ','.join([t.strip() for t in trackers_list if t.strip()])
    except:
        pass
    
    cmd = f'aria2c --seed-time=0 --summary-interval=5 --bt-enable-lpd=true --enable-dht=true -d "{temp_dir}"'
    if trackers:
        cmd += f' --bt-tracker="{trackers}"'
    cmd += f' "{VIDEO_LINK}"'
    
    os.system(cmd)
    
    # Tìm file video lớn nhất trong thư mục tải về
    video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
    video_files = []
    for root, dirs, files_in_dir in os.walk(temp_dir):
        for f in files_in_dir:
            if f.lower().endswith(video_extensions):
                full_path = os.path.join(root, f)
                video_files.append((full_path, os.path.getsize(full_path)))
    if not video_files:
        raise Exception("❌ Không tìm thấy file video nào trong Torrent tải về!")
    video_files.sort(key=lambda x: x[1], reverse=True)
    FILE_PATH = video_files[0][0]
    file_name = os.path.basename(FILE_PATH)
else:
    # --- Tải bằng yt-dlp ---
    headers = parse_headers(CUSTOM_HEADERS)
    
    ydl_opts = {
        'outtmpl': os.path.join(temp_dir, '%(title)s.%(ext)s'),
        'format': 'bestvideo+bestaudio/best',
        'merge_output_format': 'mp4',
        'quiet': False,
        'no_warnings': False,
    }
    
    if headers:
        ydl_opts['http_headers'] = headers
    
    if USE_ARIA2:
        ydl_opts['external_downloader'] = 'aria2c'
        ydl_opts['external_downloader_args'] = ['-x16', '-s16', '-k1M']
        
    print("⬇️ Bắt đầu tải video bằng yt-dlp...")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(VIDEO_LINK, download=True)
            # Lấy file video tải về
            downloaded_files = []
            for f in glob.glob(os.path.join(temp_dir, '*')):
                if not f.endswith('.part') and not f.endswith('.ytdl'):
                    downloaded_files.append(f)
            if downloaded_files:
                FILE_PATH = downloaded_files[0]
                file_name = os.path.basename(FILE_PATH)
            else:
                raise Exception("Không tìm thấy file sau khi tải.")
    except Exception as e:
        print(f"⚠️ Có lỗi khi tải bằng cấu hình tối ưu. Đang tự động chuyển sang cấu hình dự phòng...")
        # Thử tải cấu hình cơ bản không ghép kênh
        ydl_opts['format'] = 'best'
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(VIDEO_LINK, download=True)
                downloaded_files = [f for f in glob.glob(os.path.join(temp_dir, '*')) if not f.endswith('.part') and not f.endswith('.ytdl')]
                FILE_PATH = downloaded_files[0]
                file_name = os.path.basename(FILE_PATH)
        except Exception as e2:
            raise Exception(f"❌ Tải thất bại! Chi tiết lỗi: {e2}")

# Chuyển file sang Google Drive
dest_path = os.path.join(DRIVE_SAVE_PATH, file_name)
print(f"⏳ Đang di chuyển file sang Google Drive: {dest_path}...")
shutil.move(FILE_PATH, dest_path)
print(f"✅ Đã tải và lưu thành công video lên Google Drive!")
print(f"🎬 Tên file: {file_name}")
print(f"📁 Đường dẫn: {dest_path}")


In [ ]:
#@title 🧩 Bước 4: Tải & Ghép danh sách Link phân mảnh (.m4s, .ts, .mp4...)
#@markdown Dán danh sách link các phân mảnh video vào đây (mỗi link một dòng).
#@markdown 
#@markdown **Lưu ý quan trọng cho link .m4s:** Để video phát được sau khi ghép, bạn nên tìm và dán link của file khởi tạo (`init.mp4` hoặc `init.m4s` chứa metadata) vào ô `INIT_URL`. Nếu không có file init, video ghép có thể không xem được trên một số thiết bị.

INIT_URL = "" #@param {type:"string"}
SEGMENT_LINKS = "" #@param {type:"string"}
DRIVE_SAVE_PATH = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
OUTPUT_FILE_NAME = "event_video.mp4" #@param {type:"string"}
CUSTOM_HEADERS = "" #@param {type:"string"}
CONCURRENT_DOWNLOADS = 8 #@param {type:"slider", min:1, max:24, step:1}

import os
import shutil
import urllib.request
import concurrent.futures
from tqdm.notebook import tqdm

def parse_headers(headers_str):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    if not headers_str.strip():
        return headers
    for line in headers_str.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':', 1)
            headers[k.strip()] = v.strip()
    return headers

# 1. Kiểm tra và chuẩn bị danh sách link
urls = [line.strip() for line in SEGMENT_LINKS.strip().split('\n') if line.strip()]
if not urls:
    raise ValueError("❌ Danh sách SEGMENT_LINKS đang để trống!")

temp_segment_dir = "/content/temp_segments"
if os.path.exists(temp_segment_dir):
    shutil.rmtree(temp_segment_dir)
os.makedirs(temp_segment_dir, exist_ok=True)
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

headers = parse_headers(CUSTOM_HEADERS)

# Hàm tải 1 segment
def download_url(url, index, dest_dir):
    ext = "m4s"
    if ".ts" in url.lower() or "format=ts" in url.lower():
        ext = "ts"
    elif ".mp4" in url.lower():
        ext = "mp4"
    
    filename = f"segment_{index:06d}.{ext}" if index > 0 else f"init.{ext}"
    dest_file = os.path.join(dest_dir, filename)
    
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=30) as response, open(dest_file, 'wb') as f:
            f.write(response.read())
        return index, dest_file, True, None
    except Exception as e:
        return index, None, False, str(e)

# 2. Bắt đầu tải file INIT (nếu có)
init_file_path = None
if INIT_URL.strip():
    print("⏳ Đang tải file khởi tạo INIT...")
    idx, path, success, err = download_url(INIT_URL.strip(), 0, temp_segment_dir)
    if success:
        print("✅ Tải file INIT thành công!")
        init_file_path = path
    else:
        print(f"⚠️ Không thể tải file INIT ({err}). Tiếp tục tải các phân mảnh...")

# 3. Bắt đầu tải các phân mảnh song song
print(f"⬇️ Bắt đầu tải {len(urls)} phân mảnh song song (Độ chia luồng: {CONCURRENT_DOWNLOADS})...")
downloaded_paths = [None] * len(urls)
failed_count = 0

with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENT_DOWNLOADS) as executor:
    futures = {executor.submit(download_url, url, i + 1, temp_segment_dir): i for i, url in enumerate(urls)}
    
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(urls), desc="Tiến trình tải"):
        idx = futures[future]
        try:
            index, path, success, err = future.result()
            if success:
                downloaded_paths[index - 1] = path
            else:
                print(f"\n❌ Lỗi khi tải segment {index}: {err}")
                failed_count += 1
        except Exception as e:
            print(f"\n❌ Lỗi hệ thống khi tải segment {idx + 1}: {e}")
            failed_count += 1

if failed_count > 0:
    print(f"\n⚠️ Cảnh báo: Tải thất bại {failed_count}/{len(urls)} phân mảnh. Quá trình gộp có thể xảy ra lỗi.")
else:
    print("\n✅ Đã tải thành công toàn bộ phân mảnh!")

# 4. Gộp các file nhị phân (Binary Concat)
print("🔄 Đang gộp nhị phân các phân mảnh...")
raw_merged_path = "/content/temp_merged_raw.tmp"
valid_paths = [p for p in downloaded_paths if p is not None]

try:
    with open(raw_merged_path, 'wb') as outfile:
        if init_file_path and os.path.exists(init_file_path):
            with open(init_file_path, 'rb') as infile:
                outfile.write(infile.read())
        for seg_path in valid_paths:
            with open(seg_path, 'rb') as infile:
                outfile.write(infile.read())
    print("✅ Đã gộp thô các phân mảnh thành công.")
except Exception as e:
    raise Exception(f"❌ Lỗi trong quá trình gộp nhị phân: {e}")

# 5. Dùng FFmpeg để chỉnh sửa lại container và sửa lỗi timeline
print("🎬 Đang tối ưu container video bằng FFmpeg...")
local_output_path = os.path.join("/content", OUTPUT_FILE_NAME)
if os.path.exists(local_output_path):
    os.remove(local_output_path)

cmd = f'ffmpeg -i "{raw_merged_path}" -c copy -y "{local_output_path}" -v warning'
status = os.system(cmd)

if status == 0 and os.path.exists(local_output_path) and os.path.getsize(local_output_path) > 0:
    print("✅ Đã tối ưu định dạng MP4 thành công!")
    final_source_path = local_output_path
else:
    print("⚠️ FFmpeg remux thất bại. Sử dụng file gộp thô trực tiếp...")
    final_source_path = raw_merged_path
    if not OUTPUT_FILE_NAME.endswith(os.path.splitext(valid_paths[0])[1]):
        ext = os.path.splitext(valid_paths[0])[1]
        local_output_path = os.path.join("/content", os.path.splitext(OUTPUT_FILE_NAME)[0] + ext)
        final_source_path = local_output_path
    os.rename(raw_merged_path, final_source_path)

# 6. Chuyển sang Google Drive
dest_drive_path = os.path.join(DRIVE_SAVE_PATH, os.path.basename(final_source_path))
print(f"💾 Đang chuyển file kết quả sang Google Drive: {dest_drive_path}...")
shutil.move(final_source_path, dest_drive_path)
print("🎉 HOÀN THÀNH!")
print(f"📁 File đã lưu tại: {dest_drive_path}")

# Dọn dẹp
try:
    shutil.rmtree(temp_segment_dir)
    if os.path.exists(raw_merged_path): os.remove(raw_merged_path)
except:
    pass


In [ ]:
#@title 📺 Bước 5: Tải video xem lại (Catchup / Replay) từ kênh Live IPTV
#@markdown Nhập link manifest `.mpd` hoặc `.m3u8` của kênh Live và cài đặt khoảng thời gian bạn muốn xem lại (Giờ Việt Nam GMT+7).
#@markdown Sổ tay sẽ tự động kiểm tra và thêm các tham số `timeshift` để lấy luồng xem lại và tải về.

LIVE_MANIFEST_URL = "https://livevlisctcdnw.seenow.vn/livesnv2/ONSPORT6/manifest.mpd" #@param {type:"string"}
START_TIME = "2026-07-11 17:55:00" #@param {type:"string"}
END_TIME = "2026-07-11 19:00:00" #@param {type:"string"}
DRIVE_SAVE_PATH = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
OUTPUT_FILE_NAME = "event_lion_vs_wlf.mp4" #@param {type:"string"}
CUSTOM_HEADERS = "" #@param {type:"string"}

import os
import re
import glob
import shutil
import datetime
import urllib.request
import urllib.parse
import yt_dlp

def to_epoch(dt_str):
    # Chuyển đổi từ định dạng YYYY-MM-DD HH:MM:SS (GMT+7) sang Epoch
    dt = datetime.datetime.strptime(dt_str.strip(), "%Y-%m-%d %H:%M:%S")
    # Trừ đi 7 giờ để đưa về giờ quốc tế UTC
    dt_utc = dt - datetime.timedelta(hours=7)
    return int(dt_utc.timestamp())

def parse_headers(headers_str):
    headers = {}
    if not headers_str.strip():
        return headers
    for line in headers_str.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':', 1)
            headers[k.strip()] = v.strip()
    return headers

if not LIVE_MANIFEST_URL.strip():
    raise ValueError("❌ Vui lòng cung cấp link manifest của kênh live!")

t_start = to_epoch(START_TIME)
t_end = to_epoch(END_TIME)
duration_sec = t_end - t_start

if duration_sec <= 0:
    raise ValueError("❌ Thời gian bắt đầu phải nhỏ hơn thời gian kết thúc!")

print(f"⏳ Thời gian sự kiện (Giờ VN): {START_TIME} -> {END_TIME}")
print(f"⏱️ Tổng độ dài: {duration_sec} giây (~ {duration_sec/60:.1f} phút)")
print(f"🕰️ Epoch range: {t_start} -> {t_end}")

# Các kiểu tham số timeshift phổ biến trên CDN
test_templates = [
    f"?startTime={t_start}&stopTime={t_end}", # Phù hợp nhất cho Seenow/VTV Prime
    f"?start={t_start}&end={t_end}",
    f"?begin={t_start}&end={t_end}",
    f"?utc={t_start}&lutc={t_end}",
    f"?utc={t_start}000&lutc={t_end}000", # Epoch tính bằng mili giây
    f"?timeshift={int(datetime.datetime.now().timestamp() - t_start)}"
]

temp_catchup_dir = "/content/temp_catchup"
if os.path.exists(temp_catchup_dir):
    shutil.rmtree(temp_catchup_dir)
os.makedirs(temp_catchup_dir, exist_ok=True)
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

headers = parse_headers(CUSTOM_HEADERS)
parsed_url = urllib.parse.urlparse(LIVE_MANIFEST_URL)
clean_url = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}"

best_url = None

print("🔍 Đang kiểm tra các tham số Timeshift tương thích với máy chủ...")
for i, temp in enumerate(test_templates):
    target_url = clean_url + temp
    if parsed_url.query:
        target_url += "&" + parsed_url.query
        
    req = urllib.request.Request(target_url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as response:
            content = response.read().decode('utf-8', errors='ignore')
            if "SegmentTimeline" in content or "M3U" in content:
                print(f"✅ Tìm thấy phương thức phù hợp: {temp.split('=')[0]}")
                best_url = target_url
                break
    except Exception as e:
        continue

if not best_url:
    print("⚠️ Không tự động nhận diện được phản hồi Timeshift từ máy chủ.")
    print("👉 Sổ tay sẽ mặc định thử tải với phương thức ?startTime=X&stopTime=Y")
    best_url = clean_url + test_templates[0]
    if parsed_url.query:
        best_url += "&" + parsed_url.query

print(f"🔗 URL Manifest xem lại: {best_url}")

# Tải video
ydl_opts = {
    'outtmpl': os.path.join(temp_catchup_dir, 'video.%(ext)s'),
    'format': 'bestvideo+bestaudio/best',
    'merge_output_format': 'mp4',
    'quiet': False,
    'no_warnings': False,
}

if headers:
    ydl_opts['http_headers'] = headers

print("⬇️ Đang tiến hành tải video xem lại qua yt-dlp...")
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([best_url])
    
    # Tìm file đã tải
    downloaded_files = [f for f in glob.glob(os.path.join(temp_catchup_dir, '*')) if not f.endswith('.part') and not f.endswith('.ytdl')]
    if downloaded_files:
        FILE_PATH = downloaded_files[0]
        # Đổi tên và di chuyển vào Drive
        dest_path = os.path.join(DRIVE_SAVE_PATH, OUTPUT_FILE_NAME)
        print(f"💾 Đang chuyển video sang Google Drive: {dest_path}...")
        shutil.move(FILE_PATH, dest_path)
        print("🎉 TẢI XEM LẠI THÀNH CÔNG!")
        print(f"📁 Đường dẫn trên Drive: {dest_path}")
    else:
        raise Exception("Không tìm thấy video sau khi tải.")
except Exception as e:
    raise Exception(f"❌ Tải video xem lại thất bại! Chi tiết lỗi: {e}")
finally:
    try: shutil.rmtree(temp_catchup_dir)
    except: pass


In [ ]:
#@title 🎬 Bước 6: Ghép nhiều video có sẵn trên Drive / Colab
#@markdown Có 2 chế độ ghép:
#@markdown 1. **Ghép từ thư mục:** Ghép tất cả file video trong thư mục chỉ định (sắp xếp theo tên hoặc ngày tạo).
#@markdown 2. **Ghép theo danh sách:** Chỉ định chính xác các file cần ghép theo thứ tự mong muốn.

MERGE_MODE = "Gh\u00e9p t\u1eeb th\u01b0 m\u1ee5c" #@param ["Gh\u00e9p t\u1eeb th\u01b0 m\u1ee5c", "Gh\u00e9p theo danh sách file tự chọn"]
INPUT_PATH = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
DRIVE_SAVE_PATH = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
OUTPUT_FILE_NAME = "final_merged_video.mp4" #@param {type:"string"}
SORT_BY = "T\u00ean A-Z" #@param ["T\u00ean A-Z", "Th\u1eddi gian tạo (cũ -> mới)", "Th\u1eddi gian tạo (mới -> cũ)"]
RE_ENCODE = "Gh\u00e9p si\u00eau tốc (Không encode l\u1ea1i)" #@param ["Gh\u00e9p si\u00eau tốc (Không encode l\u1ea1i)", "Gh\u00e9p chất lượng (Có encode l\u1ea1i - tương th\u00edch cao)"]

import os
import glob
import shutil

# 1. Thu thập danh sách file
video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
files_to_merge = []

if MERGE_MODE == "Ghép từ thư mục":
    if not os.path.exists(INPUT_PATH):
        raise FileNotFoundError(f"❌ Không tìm thấy thư mục đầu vào: {INPUT_PATH}")
    
    all_files = []
    for f in os.listdir(INPUT_PATH):
        full_path = os.path.join(INPUT_PATH, f)
        if os.path.isfile(full_path) and f.lower().endswith(video_extensions):
            all_files.append(full_path)
            
    if not all_files:
        raise ValueError(f"❌ Không tìm thấy video nào trong thư mục: {INPUT_PATH}")
        
    if SORT_BY == "Tên A-Z":
        all_files.sort()
    elif SORT_BY == "Thời gian tạo (cũ -> mới)":
        all_files.sort(key=os.path.getctime)
    elif SORT_BY == "Thời gian tạo (mới -> cũ)":
        all_files.sort(key=os.path.getctime, reverse=True)
        
    files_to_merge = all_files
else:
    lines = [line.strip() for line in INPUT_PATH.strip().split('\n') if line.strip()]
    for line in lines:
        if os.path.exists(line):
            files_to_merge.append(line)
        else:
            print(f"⚠️ Cảnh báo: File không tồn tại và sẽ bị bỏ qua: {line}")
            
    if not files_to_merge:
        raise ValueError("❌ Không có file hợp lệ nào để tiến hành ghép!")

print(f"🎥 Tổng số file chuẩn bị ghép: {len(files_to_merge)}")
for i, f in enumerate(files_to_merge):
    print(f"  {i+1}. {os.path.basename(f)}")

# Tạo file danh sách cho FFmpeg
filelist_path = "/content/filelist.txt"
with open(filelist_path, 'w', encoding='utf-8') as f:
    for fp in files_to_merge:
        escaped_fp = fp.replace("'", "'\\''")
        f.write(f"file '{escaped_fp}'\n")

local_output_path = os.path.join("/content", OUTPUT_FILE_NAME)
if os.path.exists(local_output_path):
    os.remove(local_output_path)

os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# 2. Thực hiện ghép nối bằng FFmpeg
if RE_ENCODE == "Ghép siêu tốc (Không encode lại)":
    print("\n⚡ Đang thực hiện ghép siêu tốc (chỉ copy stream, không encode lại)...")
    cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_path}" -c copy -y "{local_output_path}" -v warning'
else:
    print("\n🎬 Đang thực hiện ghép có encode lại (sẽ chuyển đổi về H.264 + AAC)...")
    cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_path}" -vf "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2" -c:v libx264 -preset fast -crf 23 -c:a aac -b:a 192k -pix_fmt yuv420p -y "{local_output_path}" -v warning'

status = os.system(cmd)

if status == 0 and os.path.exists(local_output_path) and os.path.getsize(local_output_path) > 0:
    dest_drive_path = os.path.join(DRIVE_SAVE_PATH, OUTPUT_FILE_NAME)
    print(f"💾 Đang di chuyển file đã ghép lên Google Drive: {dest_drive_path}...")
    shutil.move(local_output_path, dest_drive_path)
    print("🎉 HOÀN THÀNH GHÉP VIDEO!")
    print(f"📁 File đã lưu tại: {dest_drive_path}")
else:
    print("❌ Lỗi xảy ra trong quá trình ghép video bằng FFmpeg.")
    if os.path.exists(local_output_path):
        try: os.remove(local_output_path)
        except: pass

if os.path.exists(filelist_path):
    os.remove(filelist_path)
